# 1주차 실습 — 토큰화와 서브워드

개념 정리는 [`notes/01-nlp-intro-tokenization.md`](../notes/01-nlp-intro-tokenization.md)를 먼저 읽고 오세요.

이 노트북에서 확인할 것:

| 실습 | 질문 |
|------|------|
| 1 | BPE는 학습에 없던 단어를 어떻게 처리하는가? |
| 2 | 어휘 크기를 바꾸면 무엇이 좋아지고 무엇이 나빠지는가? |
| 3 | 같은 의미의 문장인데 왜 한국어 토큰이 더 많은가? |
| 4 | 통계적 분절(서브워드)과 언어학적 분절(형태소)은 어떻게 다른가? |

**환경**: `conda activate nlp-study` 후 커널 `Python (nlp-study)` 선택

## 필요한 라이브러리

| 패키지 | 쓰이는 곳 |
|--------|-----------|
| `tokenizers` | 실습 1·2 — BPE 직접 학습 |
| `transformers` | 실습 3 — mBERT 토크나이저 |
| `sentencepiece` | `transformers` 토크나이저 백엔드 |
| `konlpy` | 실습 4 — 한국어 형태소 분석 |

아래 셀을 실행하면 위 패키지가 설치됩니다.

`konlpy`만은 Java 기반이라 **pip으로 해결되지 않습니다.** JDK가 없다면 터미널에서 한 번만:

```bash
conda install -n nlp-study -c conda-forge openjdk -y
```

설치 후 `conda activate nlp-study` 하면 `JAVA_HOME`이 자동으로 잡힙니다.

In [1]:
%pip install -q tokenizers transformers sentencepiece konlpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

import tokenizers
import transformers

print("python      ", sys.version.split()[0])
print("tokenizers  ", tokenizers.__version__)
print("transformers", transformers.__version__)

/Users/kimtaeyeong/miniconda3/envs/nlp-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


python       3.11.15
tokenizers   0.22.2
transformers 5.14.1


---
## 실습 1 — BPE를 직접 학습시켜 보기

노트 4.2의 손 계산 예제를 그대로 재현합니다.

```
low     × 5
lower   × 2
newest  × 6
widest  × 3
```

손으로 계산한 병합 순서는 `e+s → es`, `es+t → est`, `l+o → lo`, `lo+w → low` 였습니다.
실제로 같은 순서가 나오는지 확인합니다.

In [3]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers


def train_bpe(corpus_words, vocab_size):
    """단어 리스트로 BPE 토크나이저를 학습한다."""
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False)
    tok.train_from_iterator([" ".join(corpus_words)], trainer)
    return tok


corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
bpe = train_bpe(corpus, vocab_size=30)

# ID 순서 = 어휘에 추가된 순서 = 병합이 일어난 순서
vocab = sorted(bpe.get_vocab().items(), key=lambda kv: kv[1])
print("기본 문자:", [t for t, i in vocab if len(t) == 1])
print("병합 결과:", [t for t, i in vocab if len(t) > 1])

기본 문자: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
병합 결과: ['es', 'est', 'lo', 'low', 'ew', 'new', 'newest', 'dest', 'idest', 'widest', 'er', 'lower']


`es → est → lo → low` 순서로 병합된 것이 보입니다. 손 계산과 일치합니다.

> ⚠️ 노트 4.2에서는 Sennrich 원 논문대로 단어 끝에 `</w>`를 붙였지만,
> `tokenizers` 기본 BPE는 `</w>`를 쓰지 않습니다.
> 그래서 어휘에 `est</w>`가 아니라 `est`로 들어갑니다. **병합 순서와 최종 분절은 동일**합니다.

이제 핵심 질문입니다. **학습 코퍼스에 `lowest`는 한 번도 없었습니다.**

In [4]:
for word in ["low", "lowest", "newer", "widest", "slowest"]:
    seen = "학습됨" if word in corpus else "처음 봄"
    print(f"{word:10s} ({seen:6s}) -> {bpe.encode(word).tokens}")

low        (학습됨   ) -> ['low']
lowest     (처음 봄  ) -> ['low', 'est']
newer      (처음 봄  ) -> ['new', 'er']
widest     (학습됨   ) -> ['widest']
slowest    (처음 봄  ) -> ['s', 'low', 'est']


**확인 포인트**

- `lowest`가 `<unk>` 없이 `['low', 'est']`로 표현됩니다. 학습된 조각을 재조합한 결과입니다.
- `slowest`처럼 앞글자가 달라도 글자 단위까지 쪼개서 어떻게든 표현합니다. **OOV가 원천적으로 없습니다.**
- 이게 단어 단위 토큰화 대비 서브워드의 가장 큰 이점입니다.

---
## 실습 2 — 어휘 크기에 따른 분절 변화

어휘 크기는 **공짜 파라미터가 아닙니다.** 트레이드오프를 직접 눈으로 봅니다.

- 어휘가 작으면 → 토큰이 잘게 쪼개짐 → **시퀀스가 길어짐** → 계산량 ↑
- 어휘가 크면 → 단어가 통째로 유지됨 → **임베딩 파라미터 ↑**

In [5]:
# 실습용 소형 코퍼스 (외부 다운로드 없이 자체 포함)
text = """
natural language processing studies how computers understand human language
machine translation converts text from one language into another language
a tokenizer splits raw text into smaller units called tokens
subword tokenization balances vocabulary size and sequence length
byte pair encoding merges the most frequent adjacent pair repeatedly
neural networks learn representations directly from large amounts of data
an encoder reads the source sentence and a decoder generates the target
attention lets the decoder look back at every encoder position
the transformer replaces recurrence with self attention layers
pretraining on large corpora transfers well to downstream tasks
""".split()

corpus_large = text * 30  # 병합이 충분히 일어나도록 반복
sample = "pretraining transformers for multilingual translation"

print(f"{'vocab_size':>12} | {'토큰 수':>7} | 분절 결과")
print("-" * 90)
for size in [80, 150, 300, 1000]:
    tok = train_bpe(corpus_large, vocab_size=size)
    pieces = tok.encode(sample).tokens
    print(f"{size:>12} | {len(pieces):>7} | {pieces}")

  vocab_size |    토큰 수 | 분절 결과
------------------------------------------------------------------------------------------
          80 |      25 | ['p', 're', 'tr', 'a', 'in', 'ing', 'trans', 'f', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'trans', 'l', 'ation']
         150 |      22 | ['p', 're', 'tr', 'ain', 'ing', 'transf', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'trans', 'lation']
         300 |      17 | ['pretraining', 'transf', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'translation']
        1000 |      17 | ['pretraining', 'transf', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'translation']


**확인 포인트**

토큰 수가 `25 → 22 → 17 → 17`로 줄어듭니다. 세 가지를 함께 보세요.

**1. 어휘가 커지면 시퀀스가 짧아진다**

`pretraining`은 vocab 80에서 `p/re/tr/a/in/ing` 6조각이었다가, vocab 300에서 통째로 1토큰이 됩니다.
시퀀스가 짧아지면 계산량이 줄지만, 그 대가로 임베딩 행렬이 커집니다. **공짜가 아닙니다.**

**2. 코퍼스에 없는 단어는 어휘를 키워도 안 합쳐진다**

| 단어 | 코퍼스에 있나 | vocab 1000에서 |
|------|---------------|----------------|
| `pretraining` | ✅ 있음 | `pretraining` (1토큰) |
| `translation` | ✅ 있음 | `translation` (1토큰) |
| `transformers` | ❌ (`transformer` 단수형만 있음) | `transf/or/m/ers` (4토큰) |
| `multilingual` | ❌ 없음 | `m/u/l/t/i/l/in/gu/al` (9토큰) |

BPE는 **본 것만 합칩니다.** 도메인이 다른 텍스트를 넣으면 토큰이 급증하는 이유가 이것입니다.
실습 3에서 볼 한국어 토큰 폭증도 같은 원리입니다.

**3. vocab 300과 1000의 결과가 같다**

코퍼스가 작아서 합칠 게 더 없습니다(**어휘 포화**). 어휘 크기는 코퍼스 규모와 함께 정해야 합니다.
실제 모델이 32,000~50,000을 쓰는 것은 수억 문장 규모 코퍼스를 전제한 값입니다.

> 이 코퍼스는 실습용이라 매우 작습니다. 절대 수치보다 **변화 방향**에 주목하세요.

---
## 실습 3 — 한국어 vs 영어 토큰 효율

실제 사전학습 모델의 토크나이저(mBERT)로 같은 의미의 문장을 잘라 봅니다.

> 처음 실행하면 토크나이저 파일을 내려받습니다(수 MB). 모델 가중치는 받지 않습니다.

In [6]:
from transformers import AutoTokenizer

mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

pairs = [
    ("EN", "I really love studying natural language processing."),
    ("KO", "나는 자연어처리 공부하는 것을 정말 좋아한다."),
]

for lang, sent in pairs:
    pieces = mbert.tokenize(sent)
    print(f"[{lang}] {len(pieces):2d}토큰  {pieces}")

[EN]  8토큰  ['I', 'really', 'love', 'studying', 'natural', 'language', 'processing', '.']
[KO] 16토큰  ['나는', '자', '##연', '##어', '##처', '##리', '공', '##부', '##하는', '것을', '정', '##말', '좋', '##아', '##한다', '.']


**확인 포인트**

- 같은 의미인데 한국어 토큰 수가 영어의 **약 2배**입니다.
- `자연어처리`가 `자 / ##연 / ##어 / ##처 / ##리`로 부서집니다. 의미 단위와 전혀 맞지 않습니다.
- **왜 문제인가?** 컨텍스트 길이 제한(예: 512토큰)이 동일하므로,
  같은 제한 안에서 한국어는 **더 적은 내용**밖에 못 담습니다. API 과금도 토큰 단위입니다.
- 이 이슈는 11~12주차 다국어 모델(mBART, mT5)에서 다시 등장합니다.

---
## 실습 4 — 형태소 분석기와 비교

konlpy는 Java 기반입니다. `conda activate nlp-study` 하면 `JAVA_HOME`이 자동으로 잡힙니다.

> 첫 실행 시 JVM 관련 `WARNING: A restricted method ...` 경고가 뜹니다. 동작에는 문제 없습니다.

In [7]:
from konlpy.tag import Okt

okt = Okt()
sent_ko = "나는 자연어처리 공부하는 것을 정말 좋아한다."

print("morphs:", okt.morphs(sent_ko))
print()
print("pos   :")
for word, tag in okt.pos(sent_ko):
    print(f"    {word:6s} {tag}")

morphs: ['나', '는', '자연어', '처리', '공부', '하는', '것', '을', '정말', '좋아한다', '.']

pos   :
    나      Noun
    는      Josa
    자연어    Noun
    처리     Noun
    공부     Noun
    하는     Verb
    것      Noun
    을      Josa
    정말     Noun
    좋아한다   Adjective
    .      Punctuation


In [8]:
# 두 방식을 나란히 비교
target = "자연어처리"

print(f"'{target}' 를 어떻게 자르는가\n")
print(f"  mBERT 서브워드 (통계적) : {mbert.tokenize(target)}")
print(f"  Okt   형태소    (언어학적) : {okt.morphs(target)}")
print()

print(f"'{sent_ko}' 전체 비교\n")
print(f"  mBERT : {len(mbert.tokenize(sent_ko)):2d}개  {mbert.tokenize(sent_ko)}")
print(f"  Okt   : {len(okt.morphs(sent_ko)):2d}개  {okt.morphs(sent_ko)}")

'자연어처리' 를 어떻게 자르는가

  mBERT 서브워드 (통계적) : ['자', '##연', '##어', '##처', '##리']
  Okt   형태소    (언어학적) : ['자연어', '처리']

'나는 자연어처리 공부하는 것을 정말 좋아한다.' 전체 비교

  mBERT : 16개  ['나는', '자', '##연', '##어', '##처', '##리', '공', '##부', '##하는', '것을', '정', '##말', '좋', '##아', '##한다', '.']
  Okt   : 11개  ['나', '는', '자연어', '처리', '공부', '하는', '것', '을', '정말', '좋아한다', '.']


**확인 포인트**

| | `자연어처리` 분절 | 성격 |
|---|---|---|
| mBERT 서브워드 | `자` `##연` `##어` `##처` `##리` | **통계적** — 빈도만 보고 자름 |
| Okt 형태소 | `자연어` `처리` | **언어학적** — 의미 단위로 자름 |

형태소 분석기가 조사 `는`/`을`을 정확히 분리하고 품사까지 붙여줍니다. 언어학적으로 더 정확합니다.

**그런데 왜 요즘 모델은 형태소 분석기를 안 쓸까요?**

1. 언어마다 분석기를 따로 만들어야 합니다 → 100개 언어를 다루는 다국어 모델에선 치명적 (11주차)
2. 분석기 자체의 오류가 모델로 그대로 전파됩니다
3. 신조어·오타·구어체에 취약합니다

**확장성이 정확도를 이긴 사례**입니다. 이 관점은 커리큘럼 전체에서 반복됩니다.

---
## 정리

이 노트북에서 직접 확인한 것:

- [x] BPE는 빈도 높은 인접 쌍을 반복 병합한다 (실습 1)
- [x] 서브워드는 학습에 없던 단어도 `<unk>` 없이 표현한다 (실습 1)
- [x] 어휘 크기 ↑ → 시퀀스 길이 ↓, 임베딩 파라미터 ↑ (실습 2)
- [x] 한국어는 같은 의미를 표현하는 데 더 많은 토큰을 쓴다 (실습 3)
- [x] 통계적 분절과 언어학적 분절은 결과가 다르다 (실습 4)

**다음 주차**: 이렇게 만든 정수 ID를 의미를 담은 실수 벡터로 바꾸는 방법(Word2Vec)과,
가변 길이 시퀀스를 처리하는 RNN·LSTM을 다룹니다.